# المرحلة A: تجهيز الداتا لتجربة anomaly detection (MORE Solar Park)
الـ notebook ده بيقرا مخرجات الـ preprocessing بس، ومبيعدّلش ولا بيعيد تشغيل الـ notebook بتاعه.
المخرجات: `table_{train,val,test}.csv.gz` و`scaler_clean.json` و`inject_manifest.csv` و`dataset_summary.json`.
كل رقم في الملخص محسوب في الـ output، ومفيش موديل في المرحلة دي.

In [1]:
import gzip, hashlib, json, os, platform, random, sys
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED); np.random.seed(SEED)
print({"python": sys.version.split()[0], "pandas": pd.__version__, "numpy": np.__version__, "seed": SEED})

IN_DIR, OUT_DIR = ".", "."
SPLITS = ("train", "val", "test")
GROUPS = ("targets", "features_weather", "arrays", "data_quality_flags")
EXPECTED_ROWS = {"train": 210439, "val": 35040, "test": 35033}
STEP = pd.Timedelta("15min")
MARGIN_ROWS = 4                                   # ساعة من كل ناحية
FEATURES = ["irr_wm2", "temp_c", "mod_temp_c", "solar_elev_deg", "hour_sin", "hour_cos", "doy_sin", "doy_cos"]
TARGET = "plant_pu"
AUX = ["is_day", "wx_complete", "cand_plant_zero_daytime", "array_outage_count",
       "rule_modt_stuck", "rule_temp_sensor_episode", "rule_irr_fault_window"]
REQUIRED = {"features_weather": ["irr_wm2", "temp_c", "mod_temp_c"], "targets": [TARGET],
            "data_quality_flags": ["solar_elev_deg", "is_warmup"] + AUX}

def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

# كل الملفات المطلوبة لازم تكون موجودة (لو ناقص، الـ notebook بيقف)
need = [f"{s}_{g}.csv.gz" for s in SPLITS for g in GROUPS] + ["events.csv", "splits.json"]
missing = [n for n in need if not os.path.exists(f"{IN_DIR}/{n}")]
assert not missing, f"missing input files: {missing}"

# المدخلات لازم تكون زي ما اتكتبت (hash من تقرير الـ preprocessing لو موجود)
if os.path.exists(f"{IN_DIR}/data_quality_report.json"):
    rep = json.load(open(f"{IN_DIR}/data_quality_report.json"))["output_sha256"]
    checked = {n: (sha256_file(f"{IN_DIR}/{n}") == rep[n]) for n in need if n in rep}
    print("input hashes verified against the preprocessing report:", sum(checked.values()), "/", len(checked))
    assert all(checked.values()), f"input changed: {[n for n, ok in checked.items() if not ok]}"
INPUT_SHA = {n: sha256_file(f"{IN_DIR}/{n}") for n in need}

{'python': '3.12.3', 'pandas': '3.0.2', 'numpy': '2.4.4', 'seed': 42}
input hashes verified against the preprocessing report: 14 / 14


## A1. التحميل وإزالة الـ warm-up

In [2]:
rd = lambda n: pd.read_csv(f"{IN_DIR}/{n}", index_col="time_utc", parse_dates=True)
raw = {s: {g: rd(f"{s}_{g}.csv.gz") for g in GROUPS} for s in SPLITS}

# قائمة الأعمدة الفعلية (من ملفات train؛ ونتأكد إنها نفسها في val وtest)
for g in GROUPS:
    cols = list(raw["train"][g].columns)
    assert all(list(raw[s][g].columns) == cols for s in SPLITS), f"columns differ between splits in {g}"
    print(f"{g} ({len(cols)}):", cols)
for g, cols in REQUIRED.items():
    absent = [c for c in cols if c not in raw["train"][g].columns]
    assert not absent, f"columns missing in {g}: {absent}"

# نفس الـ index في الملفات الأربعة قبل الإزالة، ونفس الـ mask على الكل
for s in SPLITS:
    ref = raw[s]["data_quality_flags"].index
    assert all((raw[s][g].index == ref).all() for g in GROUPS), f"index differs between files in {s}"
    keep = ref[raw[s]["data_quality_flags"]["is_warmup"].to_numpy() == 0]
    n_warm = len(ref) - len(keep)
    for g in GROUPS:
        raw[s][g] = raw[s][g].loc[keep]
    print(f"{s}: rows before {len(ref):,} | warm-up removed {n_warm:,} | after {len(keep):,}")

for s in SPLITS:
    idxs = [raw[s][g].index for g in GROUPS]
    assert all(i.is_unique for i in idxs)
    assert all((i == idxs[0]).all() for i in idxs)
    assert idxs[0].is_monotonic_increasing
    assert (idxs[0].to_series().diff().dropna() == STEP).all(), f"step is not 15 min in {s}"
    assert len(idxs[0]) == EXPECTED_ROWS[s], f"unexpected row count in {s}: {len(idxs[0])} != {EXPECTED_ROWS[s]}"
print("A1 asserts passed:", {s: len(raw[s]['targets']) for s in SPLITS})

targets (1): ['plant_pu']
features_weather (9): ['temp_c', 'irr_wm2', 'mod_temp_c', 'precip_mmh', 'rh_pct', 'wdir_deg', 'wspd_ms', 'wind_u', 'wind_v']
arrays (14): ['arr1', 'arr2', 'arr3', 'arr4', 'arr5', 'arr6', 'arr7', 'arr8', 'arr9', 'arr10', 'arr11', 'arr12', 'arr13', 'arr14']
data_quality_flags (40): ['solar_elev_deg', 'is_night', 'is_day', 'rule_wspd_gt_40', 'rule_temp_sensor_episode', 'rule_modt_stuck', 'rule_irr_fault_window', 'rule_irr_gt_1400', 'wdir_wrapped', 'irr_offset_applied_wm2', 'orig_temp_c', 'orig_irr_wm2', 'orig_mod_temp_c', 'orig_wspd_ms', 'wx_all_missing_raw', 'wx_imputed_count', 'wx_unfilled_any', 'wx_complete', 'wx_complete_ex_irr', 'cand_plant_zero_daytime', 'array_outage_count', 'array_outage_arr1', 'array_outage_arr2', 'array_outage_arr3', 'array_outage_arr4', 'array_outage_arr5', 'array_outage_arr6', 'array_outage_arr7', 'array_outage_arr8', 'array_outage_arr9', 'array_outage_arr10', 'array_outage_arr11', 'array_outage_arr12', 'array_outage_arr13', 'array_ou

## A2. جدول النمذجة
الـ Features الثمانية: `irr_wm2` و`temp_c` و`mod_temp_c` من الطقس، و`solar_elev_deg` من الـ flags، وأربعة دورية من الـ index بتوقيت UTC.
`hour = index.hour + index.minute/60` و`doy = index.dayofyear`، والدورية: `sin/cos(2π·hour/24)` و`sin/cos(2π·doy/365.25)`.
`plant_pu` وأي `arr*` وأي flag **مش** في الـ input.

In [3]:
def build_table(s):
    t, w, f = raw[s]["targets"], raw[s]["features_weather"], raw[s]["data_quality_flags"]
    idx = t.index
    hour = idx.hour.to_numpy() + idx.minute.to_numpy() / 60.0
    doy = idx.dayofyear.to_numpy()
    tab = pd.DataFrame({
        "irr_wm2": w["irr_wm2"], "temp_c": w["temp_c"], "mod_temp_c": w["mod_temp_c"],
        "solar_elev_deg": f["solar_elev_deg"],
        "hour_sin": np.sin(2 * np.pi * hour / 24.0), "hour_cos": np.cos(2 * np.pi * hour / 24.0),
        "doy_sin": np.sin(2 * np.pi * doy / 365.25), "doy_cos": np.cos(2 * np.pi * doy / 365.25),
        TARGET: t[TARGET],
    }, index=idx)
    for c in AUX:
        tab[c] = f[c]
    return tab

tables = {s: build_table(s) for s in SPLITS}
assert list(tables["train"].columns[:8]) == FEATURES
forbidden = [c for c in FEATURES if c == TARGET or c.startswith("arr") or c.startswith(("rule_", "cand_", "wx_", "is_"))]
assert not forbidden and len(FEATURES) == 8
print("model inputs (8):", FEATURES)
print("target:", TARGET, "| aux (not inputs):", AUX)
for s in SPLITS:
    assert (tables[s]["is_day"].to_numpy() == (tables[s]["solar_elev_deg"].to_numpy() > 10).astype(int)).all()
print("A2 asserts passed (is_day == solar_elev_deg > 10 in all splits)")

model inputs (8): ['irr_wm2', 'temp_c', 'mod_temp_c', 'solar_elev_deg', 'hour_sin', 'hour_cos', 'doy_sin', 'doy_cos']
target: plant_pu | aux (not inputs): ['is_day', 'wx_complete', 'cand_plant_zero_daytime', 'array_outage_count', 'rule_modt_stuck', 'rule_temp_sensor_episode', 'rule_irr_fault_window']
A2 asserts passed (is_day == solar_elev_deg > 10 in all splits)


## A3. الصفوف القابلة للتقييم
`scoreable = (is_day==1) & (wx_complete==1)` ومعاهم الـ 8 features والـ target مش NaN.

In [4]:
for s in SPLITS:
    t = tables[s]
    t["scoreable"] = ((t["is_day"] == 1) & (t["wx_complete"] == 1)
                      & t[FEATURES].notna().all(axis=1) & t[TARGET].notna()).astype(int)

def cov_row(label, t):
    d, sc = int((t["is_day"] == 1).sum()), int(t["scoreable"].sum())
    return {"split": label, "rows": len(t), "is_day rows": d, "scoreable rows": sc, "coverage %": round(100 * sc / d, 2)}

rows = [cov_row(s, tables[s]) for s in SPLITS]
CUT = pd.Timestamp("2023-10-12", tz="UTC")
tt = tables["test"]
rows += [cov_row("test (< 2023-10-12)", tt[tt.index < CUT]), cov_row("test (>= 2023-10-12)", tt[tt.index >= CUT])]
a3 = pd.DataFrame(rows)
print(a3.to_string(index=False))

               split   rows  is_day rows  scoreable rows  coverage %
               train 210439        88943           86985       97.80
                 val  35040        14815           12214       82.44
                test  35033        14812           12180       82.23
 test (< 2023-10-12)  27264        12295           12180       99.06
test (>= 2023-10-12)   7769         2517               0        0.00


## A4. الـ clean set
`clean = scoreable & (cand_plant_zero_daytime==0) & (array_outage_count==0) & rule flags الثلاثة == 0` وكمان مش جوا أي حدث حقيقي موسّع.
الأحداث: `plant_zero_daytime` و`array_outage` من `events.csv`، وكل حدث بيتوسّع بـ 4 صفوف (ساعة) من الناحيتين.
`in_event_margin = 1` لكل صف جوا المدى الموسّع (الحدث نفسه والهوامش).

In [5]:
ev = pd.read_csv(f"{IN_DIR}/events.csv")
ev["start_utc"] = pd.to_datetime(ev["start_utc"], utc=True)
ev["end_utc"] = pd.to_datetime(ev["end_utc"], utc=True)
real = ev[ev["type"].isin(["plant_zero_daytime", "array_outage"])].reset_index(drop=True)
print("events in file:", ev["type"].value_counts().to_dict())
print("events used for margins:", real["type"].value_counts().to_dict())

def mark_margin(index, events, margin_rows=MARGIN_ROWS):
    vals = index.asi8
    lo = (events["start_utc"] - margin_rows * STEP).astype("int64").to_numpy()
    hi = (events["end_utc"] + margin_rows * STEP).astype("int64").to_numpy()
    # الـ index و events بنفس وحدة الزمن
    unit = index.dtype.unit
    scale = {"ns": 1, "us": 1000, "ms": 10**6, "s": 10**9}[unit]
    a = np.searchsorted(vals, (events["start_utc"] - margin_rows * STEP).map(lambda x: x.value // scale).to_numpy(), side="left")
    b = np.searchsorted(vals, (events["end_utc"] + margin_rows * STEP).map(lambda x: x.value // scale).to_numpy(), side="right")
    delta = np.zeros(len(index) + 1, dtype=int)
    np.add.at(delta, a, 1); np.add.at(delta, b, -1)
    return (np.cumsum(delta)[:-1] > 0)

for s in SPLITS:
    t = tables[s]
    t["in_event_margin"] = mark_margin(t.index, real).astype(int)
    t["clean"] = ((t["scoreable"] == 1) & (t["cand_plant_zero_daytime"] == 0) & (t["array_outage_count"] == 0)
                  & (t["rule_modt_stuck"] == 0) & (t["rule_temp_sensor_episode"] == 0)
                  & (t["rule_irr_fault_window"] == 0) & (t["in_event_margin"] == 0)).astype(int)
    t["split"] = s

a4 = pd.DataFrame([{"split": s, "scoreable": int(tables[s]["scoreable"].sum()), "in_event_margin rows": int(tables[s]["in_event_margin"].sum()),
                    "clean rows": int(tables[s]["clean"].sum())} for s in SPLITS])
print(a4.to_string(index=False))
assert tables["train"]["clean"].sum() >= 20000, "STOP: train clean < 20,000"
assert tables["val"]["clean"].sum() >= 5000, "STOP: val clean < 5,000"
assert all(((tables[s]["clean"] == 1) <= (tables[s]["scoreable"] == 1)).all() for s in SPLITS)

# فترة تصحيح إزاحة الإشعاع بتفضل جوا clean
OFF_A, OFF_B = pd.Timestamp("2022-08-21", tz="UTC"), pd.Timestamp("2023-10-12", tz="UTC")
offset_clean = {s: int(tables[s]["clean"][(tables[s].index >= OFF_A) & (tables[s].index < OFF_B)].sum()) for s in SPLITS}
print("clean rows inside the irradiance-offset correction period (2022-08-21 .. 2023-10-11):", offset_clean)

events in file: {'array_outage': 271, 'plant_zero_daytime': 49, 'weather_all_missing': 24, 'weather_partial_missing': 7, 'irradiance_sensor_fault': 3, 'module_temp_stuck': 1, 'temp_sensor_episode': 1, 'irradiance_offset': 1}
events used for margins: {'array_outage': 271, 'plant_zero_daytime': 49}
split  scoreable  in_event_margin rows  clean rows
train      86985                  3653       83135
  val      12214                   121       12146
 test      12180                  3356        8857
clean rows inside the irradiance-offset correction period (2022-08-21 .. 2023-10-11): {'train': 0, 'val': 4659, 'test': 8857}


In [6]:
# الـ scaler من clean train بس: median و IQR (أو std لو IQR = 0)
tr = tables["train"]; trc = tr[tr["clean"] == 1]
params = {}
for c in FEATURES + [TARGET]:
    x = trc[c].dropna()
    q25, med, q75 = (float(v) for v in x.quantile([0.25, 0.5, 0.75]))
    iqr, std = q75 - q25, float(x.std())
    scale, rule = (iqr, "iqr") if iqr > 0 else (std, "std_fallback")
    params[c] = {"median": med, "q25": q25, "q75": q75, "iqr": iqr, "std": std, "scale": scale, "scale_rule": rule}
    assert scale > 0
scaler_clean = {"method": "(x - median) / scale", "fit_set": "clean rows of train only",
                "fit_rows": int(len(trc)), "fit_window": {"start_utc": str(trc.index.min()), "end_utc": str(trc.index.max())},
                "columns": FEATURES + [TARGET], "params": params}
json.dump(scaler_clean, open(f"{OUT_DIR}/scaler_clean.json", "w"), indent=2, sort_keys=True)
print(pd.DataFrame(params).T[["median", "iqr", "std", "scale", "scale_rule"]].round(5).to_string())

                  median       iqr         std     scale scale_rule
irr_wm2         508.5056  546.9669  303.060369  546.9669        iqr
temp_c           19.7483   10.0779    6.188839   10.0779        iqr
mod_temp_c       26.8559  14.02445    9.154289  14.02445        iqr
solar_elev_deg    32.453   25.8665   16.658334   25.8665        iqr
hour_sin        0.321439  1.187465    0.624227  1.187465        iqr
hour_cos       -0.793353  0.504641     0.34589  0.504641        iqr
doy_sin         0.096613  1.430938    0.708724  1.430938        iqr
doy_cos        -0.283139  1.308668    0.688369  1.308668        iqr
plant_pu          0.4345    0.4501    0.244521    0.4501        iqr


## A5. مجموعة الحقن الصناعي (على test فقط)
الأيام المؤهلة: أيام UTC في test مفيهاش أي صف جوا أحداث حقيقية موسّعة، وفيها `scoreable` عند 80% أو أكتر من صفوف `solar_elev_deg>10`.
الحدث لازم يكون كله جوا صفوف `scoreable` و`solar_elev_deg>10` ومتصلة في نفس اليوم.
اختيار الأيام وبداية الحدث بـ `default_rng([42, cell_index])`، وبداية الحدث عشوائية من المواضع الممكنة في اليوم.

In [7]:
tt = tables["test"].copy()
assert (tt.index.to_series().diff().dropna() == STEP).all()
tt["day"] = tt.index.normalize()
daymask = tt["solar_elev_deg"] > 10
tt["ok"] = ((tt["scoreable"] == 1) & daymask).to_numpy()

g = tt.groupby("day")
day_tbl = pd.DataFrame({
    "rows": g.size(),
    "event_rows": g["in_event_margin"].sum(),
    "day_rows": daymask.groupby(tt["day"]).sum(),
    "scoreable_day_rows": (tt["scoreable"].astype(bool) & daymask).groupby(tt["day"]).sum(),
})
day_tbl["coverage"] = (day_tbl["scoreable_day_rows"] / day_tbl["day_rows"]).where(day_tbl["day_rows"] > 0)
day_tbl["no_event"] = day_tbl["event_rows"] == 0
day_tbl["cov_ok"] = day_tbl["coverage"] >= 0.8
day_tbl["eligible"] = day_tbl["no_event"] & day_tbl["cov_ok"]
print("test UTC days:", len(day_tbl), "| no real event:", int(day_tbl["no_event"].sum()),
      "| coverage >= 80%:", int(day_tbl["cov_ok"].sum()), "| eligible (both):", int(day_tbl["eligible"].sum()))

DURS = {1: 4, 3: 12, 6: 24}      # ساعات -> صفوف
ok_by_day = {d: x["ok"].to_numpy() for d, x in tt.groupby("day")}
idx_by_day = {d: x.index for d, x in tt.groupby("day")}

def feasible_starts(ok, n):
    if len(ok) < n:
        return np.array([], dtype=int)
    cs = np.concatenate([[0], np.cumsum(ok.astype(int))])
    return np.flatnonzero((cs[n:] - cs[:-n]) == n)

elig_days = list(day_tbl.index[day_tbl["eligible"]])
cap_rows = []
for h, n in DURS.items():
    able = [d for d in elig_days if len(feasible_starts(ok_by_day[d], n)) > 0]
    cap_rows.append({"duration_h": h, "eligible days (total)": len(elig_days), "days that fit the event": len(able)})
cap = pd.DataFrame(cap_rows)
print(cap.to_string(index=False))

test UTC days: 365 | no real event: 279 | coverage >= 80%: 280 | eligible (both): 199


 duration_h  eligible days (total)  days that fit the event
          1                    199                      199
          3                    199                      199
          6                    199                      199

In [8]:
CELLS = [("drop", s, h) for s in (0.10, 0.20, 0.40, 1.00) for h in (1, 3, 6)] + [("stuck", np.nan, h) for h in (1, 3, 6)]
N_PER_CELL = 25
test_plant = tt[TARGET].to_numpy()

def make_manifest():
    recs, starts_log = [], {}
    for ci, (typ, sev, h) in enumerate(CELLS):
        n = DURS[h]
        rng = np.random.default_rng([SEED, ci])
        able = [d for d in elig_days if len(feasible_starts(ok_by_day[d], n)) > 0]
        k = min(N_PER_CELL, len(able))
        chosen = sorted(rng.choice(len(able), size=k, replace=False)) if k > 0 else []
        for j in chosen:
            d = able[j]
            fs = feasible_starts(ok_by_day[d], n)
            st = int(rng.choice(fs))
            ix = idx_by_day[d]
            sev_tag = "NA" if typ == "stuck" else f"s{int(round(sev * 100)):03d}"
            recs.append({"event_id": f"{typ}_{sev_tag}_{h}h_{d.date()}", "type": typ, "severity": sev, "duration_h": h,
                         "day": str(d.date()), "start_utc": ix[st], "end_utc": ix[st + n - 1]})
    return pd.DataFrame(recs)

manifest = make_manifest()
assert manifest.equals(make_manifest()), "manifest is not reproducible"
assert manifest["event_id"].is_unique
assert set(zip(manifest["type"], manifest["severity"].fillna(-1), manifest["duration_h"])) <= set((a, -1 if np.isnan(b) else b, c) for a, b, c in CELLS)

# كل حدث كله جوا صفوف scoreable ونهارية ومتصلة وخارج أي حدث حقيقي موسّع
pos = {t: i for i, t in enumerate(tt.index)}
for r in manifest.itertuples():
    a, b = pos[r.start_utc], pos[r.end_utc]
    seg = tt.iloc[a:b + 1]
    assert len(seg) == DURS[r.duration_h] and seg["ok"].all() and (seg["in_event_margin"] == 0).all()
    assert seg.index[0].normalize() == seg.index[-1].normalize()

per_cell = (manifest.assign(sev=manifest["severity"].fillna(-1)).groupby(["type", "sev", "duration_h"]).size()
            .rename("events").reset_index())
per_cell["sev"] = per_cell["sev"].replace(-1, np.nan)
print(per_cell.to_string(index=False))
print("total events:", len(manifest), "| cells:", len(per_cell), "| max per day-cell pair:",
      int(manifest.groupby(["type", "severity", "duration_h", "day"], dropna=False).size().max()))
manifest.to_csv(f"{OUT_DIR}/inject_manifest.csv", index=False,
                columns=["event_id", "type", "severity", "duration_h", "day", "start_utc", "end_utc"])

 type  sev  duration_h  events
 drop  0.1           1      25
 drop  0.1           3      25
 drop  0.1           6      25
 drop  0.2           1      25
 drop  0.2           3      25
 drop  0.2           6      25
 drop  0.4           1      25
 drop  0.4           3      25
 drop  0.4           6      25
 drop  1.0           1      25
 drop  1.0           3      25
 drop  1.0           6      25
stuck  NaN           1      25
stuck  NaN           3      25
stuck  NaN           6      25
total events: 375 | cells: 15 | max per day-cell pair: 1


In [9]:
# حجم التغيير الفعلي (p.u.) لكل خلية: متوسط |plant' - plant| داخل الحدث (للتقرير فقط، مش بيغيّر الـ manifest)
changes = []
for r in manifest.itertuples():
    a, b = pos[r.start_utc], pos[r.end_utc]
    seg = test_plant[a:b + 1]
    delta = seg * r.severity if r.type == "drop" else np.abs(seg - seg[0])
    changes.append(float(np.mean(np.abs(delta))))
manifest_chg = manifest.assign(sev=manifest["severity"].fillna(-1), mean_abs_change_pu=changes)
chg = (manifest_chg.groupby(["type", "sev", "duration_h"])["mean_abs_change_pu"].agg(["count", "median"]).round(4).reset_index())
chg["sev"] = chg["sev"].replace(-1, np.nan)
print(chg.rename(columns={"count": "events", "median": "median mean_abs_change_pu"}).to_string(index=False))

 type  sev  duration_h  events  median mean_abs_change_pu
 drop  0.1           1      25                     0.0408
 drop  0.1           3      25                     0.0566
 drop  0.1           6      25                     0.0429
 drop  0.2           1      25                     0.0867
 drop  0.2           3      25                     0.1087
 drop  0.2           6      25                     0.1008
 drop  0.4           1      25                     0.2163
 drop  0.4           3      25                     0.1897
 drop  0.4           6      25                     0.2077
 drop  1.0           1      25                     0.4768
 drop  1.0           3      25                     0.5324
 drop  1.0           6      25                     0.5702
stuck  NaN           1      25                     0.0632
stuck  NaN           3      25                     0.1491
stuck  NaN           6      25                     0.1782


## A6. المخرجات

In [10]:
OUT_COLS = FEATURES + [TARGET, "scoreable", "clean", "in_event_margin", "split"]
def write_table(s):
    t = tables[s][OUT_COLS]
    txt = t.to_csv(index_label="time_utc", float_format="%.6f")
    with open(f"{OUT_DIR}/table_{s}.csv.gz", "wb") as fh:
        with gzip.GzipFile(filename="", mode="wb", fileobj=fh, mtime=0) as gz:
            gz.write(txt.encode())
for s in SPLITS:
    write_table(s)

summary = {
    "environment": {"python": sys.version.split()[0], "pandas": pd.__version__, "numpy": np.__version__, "seed": SEED},
    "input_sha256": INPUT_SHA,
    "features": FEATURES, "target": TARGET, "output_columns": OUT_COLS,
    "rows_after_warmup_removal": {s: len(tables[s]) for s in SPLITS},
    "coverage_table": a3.to_dict(orient="records"),
    "clean_table": a4.to_dict(orient="records"),
    "events_used_for_margins": real["type"].value_counts().to_dict(), "margin_rows_each_side": MARGIN_ROWS,
    "scaler_fit_rows": int(len(trc)),
    "test_days": {"total": int(len(day_tbl)), "no_real_event": int(day_tbl["no_event"].sum()),
                  "coverage_ge_80": int(day_tbl["cov_ok"].sum()), "eligible": int(day_tbl["eligible"].sum())},
    "eligible_days_by_duration": cap.to_dict(orient="records"),
    "injection_cells": per_cell.assign(sev=per_cell["sev"].astype(object).where(per_cell["sev"].notna(), None)).to_dict(orient="records"),
    "injection_events_total": int(len(manifest)),
    "injection_change_size_pu": chg.assign(sev=chg["sev"].astype(object).where(chg["sev"].notna(), None)).to_dict(orient="records"),
    "notes": [
        "Irradiance-offset correction period (2022-08-21 .. 2023-10-11) stays inside clean; clean rows inside it: " + json.dumps(offset_clean),
        "in_event_margin marks every row inside plant_zero_daytime/array_outage events expanded by 4 rows on each side (event rows included).",
        "hour/doy features use UTC; doy period 365.25 days.",
        "Injection start inside a day is random among feasible positions (default_rng([42, cell_index])); severity is blank for stuck.",
        "Scaler uses clean train rows only; scaler_params.json from the preprocessing step is not used.",
    ],
}
summary["output_sha256"] = {n: sha256_file(f"{OUT_DIR}/{n}") for n in ("table_train.csv.gz", "table_val.csv.gz", "table_test.csv.gz", "scaler_clean.json", "inject_manifest.csv")}
json.dump(summary, open(f"{OUT_DIR}/dataset_summary.json", "w"), indent=2, sort_keys=True, default=lambda o: o.item() if hasattr(o, "item") else str(o))

# تحقق نهائي من الملفات المكتوبة
for s in SPLITS:
    t = pd.read_csv(f"{OUT_DIR}/table_{s}.csv.gz", index_col="time_utc", parse_dates=True)
    assert len(t) == EXPECTED_ROWS[s] and list(t.columns) == OUT_COLS
    assert (t["split"] == s).all() and t.index.is_unique
    assert ((t["clean"] == 1) <= (t["scoreable"] == 1)).all()
    assert t.loc[t["scoreable"] == 1, FEATURES + [TARGET]].notna().all().all()
m = pd.read_csv(f"{OUT_DIR}/inject_manifest.csv")
assert list(m.columns) == ["event_id", "type", "severity", "duration_h", "day", "start_utc", "end_utc"] and len(m) == len(manifest)
print("A6 asserts passed")
for n in sorted(summary["output_sha256"]):
    print(f"{n:24s} {os.path.getsize(f'{OUT_DIR}/{n}'):>12,d} bytes")
print("dataset_summary.json", f"{os.path.getsize(f'{OUT_DIR}/dataset_summary.json'):,d}", "bytes")

A6 asserts passed
inject_manifest.csv            36,581 bytes
scaler_clean.json               2,503 bytes
table_test.csv.gz             790,723 bytes
table_train.csv.gz          4,745,584 bytes
table_val.csv.gz              763,891 bytes
dataset_summary.json 8,193 bytes
